In [1]:
import pandas as pd
import numpy as np
import pydicom
import cv2
import os
import shutil

data_dir = "/kaggle/input/competitions/rsna-knee-abnormality-detection"

train = pd.read_csv(f"{data_dir}/train.csv")
train_series = pd.read_csv(f"{data_dir}/train_series.csv")

finding_columns = ['ACL', 'MCL', 'Medial Meniscus', 'Lateral Meniscus', 'Medial OA', 
                    'Lateral OA', 'PF OA', 'Effusion', 'Synovitis', "Baker's", 
                    'Contusion', 'Fracture']

gold_data = train[train['Fracture'].notna()]
gold_labels = gold_data[['StudyInstanceUID'] + finding_columns].copy()
report_label = pd.read_csv("/kaggle/input/datasets/mayankptdr/report-labels/extracted_labels.csv")
all_labels = pd.concat([gold_labels, report_label], ignore_index=True)
all_labels[finding_columns] = all_labels[finding_columns].fillna(0)

SLOTS = [('Sagittal', 1), ('Sagittal', 0), ('Coronal', 1), ('Coronal', 0), ('Axial', 1)]

def select_series_5slots(study_id, train_series_df):
    study_series = train_series_df[train_series_df['StudyInstanceUID'] == study_id]
    slots = {}
    for plane, fluid_sensitive in SLOTS:
        key = f"{plane}_{fluid_sensitive}"
        candidates = study_series[
            (study_series['Anatomical_Plane'] == plane) & 
            (study_series['Fluid_Sensitive'] == fluid_sensitive)
        ]
        slots[key] = candidates.iloc[0]['SeriesInstanceUID'] if len(candidates) > 0 else None
    return slots

def select_slice_indices(total_slices, n_slice=16, band=(0.12, 0.88)):
    start = int(total_slices * band[0])
    end = int(total_slices * band[1])
    indices = np.linspace(start, end-1, n_slice).astype(int)
    return indices

def crop_by_mm(pixel_array, pixel_spacing, crop_mm=130.0):
    row_spacing, col_spacing = pixel_spacing
    crop_h = int(crop_mm / row_spacing)
    crop_w = int(crop_mm / col_spacing)
    h, w = pixel_array.shape
    center_h, center_w = h // 2, w // 2
    half_h, half_w = crop_h // 2, crop_w // 2
    top = max(0, center_h - half_h)
    bottom = min(h, center_h + half_h)
    left = max(0, center_w - half_w)
    right = min(w, center_w + half_w)
    return pixel_array[top:bottom, left:right]

def crop_and_resize(pixel_array, pixel_spacing, crop_mm=130.0, final_size=224):
    cropped = crop_by_mm(pixel_array, pixel_spacing, crop_mm)
    resized = cv2.resize(cropped, (final_size, final_size), interpolation=cv2.INTER_LINEAR)
    return resized

print(f"Setup complete. Total studies: {all_labels.shape[0]}")

Setup complete. Total studies: 4407


In [2]:
CHUNK_NUMBER = 1 

all_study_ids = all_labels['StudyInstanceUID'].tolist()
chunk_size = len(all_study_ids) // 5

chunks = {
    1: all_study_ids[:chunk_size],
    2: all_study_ids[chunk_size:2*chunk_size],
    3: all_study_ids[2*chunk_size:3*chunk_size],
    4: all_study_ids[3*chunk_size:4*chunk_size],
    5: all_study_ids[4*chunk_size:]
}

my_chunk = chunks[CHUNK_NUMBER]
print(f"Chunk {CHUNK_NUMBER}: {len(my_chunk)} studies assigned")

Chunk 1: 881 studies assigned


In [3]:
cache_dir_dino = "/kaggle/working/cached_dino"
os.makedirs(cache_dir_dino, exist_ok=True)

def build_cache_dino(study_ids):
    for study_id in study_ids:
        cache_path = f"{cache_dir_dino}/{study_id}.npz"
        if os.path.exists(cache_path):
            continue
        
        total, used, free = shutil.disk_usage("/kaggle/working")
        if free / (1024**3) < 1.0:
            print(f"Disk space low. Stopping safely.")
            break
        
        try:
            slot_map = select_series_5slots(study_id, train_series)
            slot_volumes, slot_masks = [], []
            
            for plane, fs in SLOTS:
                key = f"{plane}_{fs}"
                series_id = slot_map[key]
                
                if series_id is None:
                    vol = np.zeros((16, 224, 224), dtype=np.float16)
                    mask = 0.0
                else:
                    folder = f"{data_dir}/train_series/{study_id}/{series_id}"
                    dcm_files = os.listdir(folder)
                    slices = [pydicom.dcmread(f"{folder}/{f}") for f in dcm_files]
                    slices_sorted = sorted(slices, key=lambda d: int(d.InstanceNumber))
                    total_slices = len(slices_sorted)
                    indices = select_slice_indices(total_slices, 16)
                    pixel_spacing = [float(x) for x in slices_sorted[0].PixelSpacing]
                    
                    processed = []
                    for idx in indices:
                        arr = slices_sorted[idx].pixel_array.astype(np.float32)
                        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-6)
                        cr = crop_and_resize(arr, pixel_spacing, 130.0, 224)
                        processed.append(cr)
                    vol = np.stack(processed).astype(np.float16)
                    mask = 1.0
                
                slot_volumes.append(vol)
                slot_masks.append(mask)
            
            slots_array = np.stack(slot_volumes)
            mask_array = np.array(slot_masks, dtype=np.float32)
            np.savez_compressed(cache_path, slots=slots_array, mask=mask_array)
            print(f"Cached: {study_id}")
        
        except Exception as e:
            print(f"Error on {study_id}: {e}")

build_cache_dino(my_chunk)

Cached: 1.2.826.0.1.3680043.8.498.10095687747295410396510538520594649149
Cached: 1.2.826.0.1.3680043.8.498.10170898615867673028696505248839028269
Cached: 1.2.826.0.1.3680043.8.498.10306159113324811538703788080836752052
Cached: 1.2.826.0.1.3680043.8.498.11287937729196958426538087439102017580
Cached: 1.2.826.0.1.3680043.8.498.11382021393803389951964005983002209238
Cached: 1.2.826.0.1.3680043.8.498.11548045715264151632153040089882701935
Cached: 1.2.826.0.1.3680043.8.498.11557620559191469069130827959098335840
Cached: 1.2.826.0.1.3680043.8.498.11771393824519892797114773408583976756
Cached: 1.2.826.0.1.3680043.8.498.11851412923016044948101698015974810604
Cached: 1.2.826.0.1.3680043.8.498.11915937982684988073644209606907169581
Cached: 1.2.826.0.1.3680043.8.498.12448079646359892252441208258836556945
Cached: 1.2.826.0.1.3680043.8.498.12505035424093604269515328931488770819
Cached: 1.2.826.0.1.3680043.8.498.12606657226568558340797193167488111973
Cached: 1.2.826.0.1.3680043.8.498.12801308844398614